# 位置编码

---

## 一、正弦位置编码
> Attention is all you need (NIPS2017)

一维位置编码公式如下:
$$
\begin{align}
PE(pos, 2i) &= \sin(pos / 10000^{2i/d_{model}}) \\
PE(pos, 2i+1) &= \cos(pos / 10000^{2i/d_{model}})
\end{align}
$$
也就是说，对于偶数维位置用sin编码，对于奇数维位置用cos编码

这个编码是常量 (不可学习), 可以预先计算出来

In [1]:
import torch

In [2]:
def create_position_embedding(n_pos_vec, dim):
    """
    n_pos_vec: torch.arange(0, n_pos)
    dim: embedding dimension of each row
    """
    
    # 首先保证嵌入维度必须是偶数
    assert dim % 2 == 0, "dim must be even"
    
    """
    初始化嵌入矩阵 (全0矩阵即可), 形状为 (num_pos, dim)
    numel: 矩阵中元素的总数
    """
    position_embedding = torch.zeros((n_pos_vec.numel(), dim), dtype=torch.float32)
    
    """
    omega对i进行遍历
    假设i=0,1,2,3
    那么偶数列(2i)=0,2,4,6
    那么奇数列(2i+1)=1,3,5,7
    总体来看, i可以用以表示所有的维度位置(0~7)
    同时无论奇偶列, 其嵌入值都是偶数2i
    """
    omega = torch.arange(dim/2, dtype=torch.float32)
    
    # 计算三角函数内的指数部分, 记作omega
    omega /= dim / 2                # i/(dim/2)
    omega = 1.0 / (10000 ** omega)  # 1/(10000^(i/(dim/2)))
    
    """
    利用**矩阵乘法**，计算三角函数内的乘积值
    None: unsqueeze
    列向量 @ 行向量 = 矩阵
    """
    out = n_pos_vec[:, None] @ omega[None, :]
    
    """
    分别计算偶数维sin嵌入+奇数维cos嵌入
    偶数维: 0,2,4,6; 奇数维: 1,3,5,7
    两个形状都是(num_pos, dim/2)
    """
    emb_sin = torch.sin(out)
    emb_cos = torch.cos(out)
    
    # sin值放pe偶数维, cos值放pe奇数维, 奇偶交替
    position_embedding[:, 0::2] = emb_sin
    position_embedding[:, 1::2] = emb_cos
    
    return position_embedding
    

In [3]:
# 4个位置, 嵌入维度为8
n_pos_vec = torch.arange(4, dtype=torch.float32)
dim = 8
pe = create_position_embedding(n_pos_vec, dim)

In [4]:
pe

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00]])

### 正弦编码的部分结论
- 线性的相对位置表达能力: 对于固定位置间距$k$, $PE(pos+k)$可以由$PE(pos)$线性表出, 以偶数列(2i)为例:
    $$
    \begin{align}
    PE(pos+k, 2i) &= \sin(pos * \omega_{2i} + k * \omega_{2i}) \\
                  &= \sin(pos * \omega_{2i})\cos(k * \omega_{2i}) + \cos(pos * \omega_{2i})\sin(k * \omega_{2i}) \\
                  &= PE(pos, 2i) \cos(k * \omega_{2i}) + PE(pos, 2i+1) \sin(k * \omega_{2i})
    \end{align}
    $$
    即直接用**和角公式**展开$PE(pos+k, 2i)$; 其中$\omega_{2i}=\dfrac{1}{10000^{2i/d_{model}}}$
    
- 内积之与相对位置$k$有关: 两个位置的内积表示形式如下:
    $$
    \begin{align}
    PE(pos)PE(pos+k) &= \sum_{i=0}^{d/2-1}PE(pos, 2i)PE(pos+k, 2i) + \sum_{i=0}^{d/2-1}PE(pos, 2i+1)PE(pos+k, 2i+1) \\
                     &= \sum_{i=0}^{d/2-1}\sin(pos * \omega_{2i})\sin[(pos+k) * \omega_{2i}] + \cos(pos * \omega_{2i})\cos[(pos+k) * \omega_{2i}] \\
                     &= \sum_{i=0}^{d/2-1} \cos(k * \omega_{2i})
    \end{align}
    $$
    偶数列和偶数列内积, 奇数列和奇数列内积, 再用$\cos$和角公式合并; 可以看出内积结果只与$k$有关, 且随$k$的绝对值增加而减小

- 为什么底数base要用10000? 底数越大, $1/base^{2i/d}$越小, 关于自变量$pos$的周期越大, 保证了跨越多个$pos$后尽量不出现重复的$PE(pos)$

## 二、ViT位置编码
> TO BE DONE